In [2]:
import uiautomator2 as u2
# 连接并启动
d = u2.connect_usb()
print(d.info)
print(d.serial)
# print(d.dump_hierarchy())

The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.
{'currentPackageName': 'com.xingin.xhs', 'displayHeight': 2756, 'displayRotation': 0, 'displaySizeDpX': 390, 'displaySizeDpY': 848, 'displayWidth': 1268, 'productName': 'klee', 'screenOn': True, 'sdkInt': 36, 'naturalOrientation': True}
996HYXX8XW99BE7P


In [ ]:
import random
import time
from bs4 import BeautifulSoup
from bs4 import Tag
from typing import List
import sqlite3
import re

soup = BeautifulSoup(d.dump_hierarchy(), "xml")
nodes = soup.find_all("node", attrs={"class": "android.widget.FrameLayout"})


def parse_bounds(bounds: str) -> tuple[int, int, int, int]:
    """解析 bounds 字符串为 (left, top, right, bottom) 元组。"""
    x1, y1, x2, y2 = map(int, bounds.strip("[]").replace("][", ",").split(","))
    return x1, y1, x2, y2

def swiper_up():
    """向上滑动。"""
    width, height = d.window_size()
    sx = width // 2             # 水平方向中点
    sy = height // 2            # 垂直方向中点（起点）
    ey = sy - height // 2       # 向上半屏
    d.swipe(sx, sy, sx, ey, 0.5)  # 0.5秒滑动
    time.sleep(random.uniform(1, 2))

def parse_note() -> dict | None:
    """解析笔记节点。"""
    # 先找到目标节点

    def _parse_note_title_content_author():
        soup = BeautifulSoup(d.dump_hierarchy(), "xml")
        """解析笔记标题、内容和作者。"""
        NOTE_MAP = {
            "author": {
                "resource-id": "com.xingin.xhs:id/nickNameTV",
            },
            "content": {
                "class": "android.widget.TextView",
                "drawing-order": "3",
                "resource-id": "com.xingin.xhs:id/0_resource_name_obfuscated",
            }
        }

        author_text = soup.find(name="node", attrs=NOTE_MAP["author"]).get("text", None)

        content_node = soup.find(
           name= "node",
            attrs= NOTE_MAP["content"],
        )

        if not content_node:
            swiper_up()

            soup = BeautifulSoup(d.dump_hierarchy(), "xml")
            content_node = soup.find(
                "node",
                attrs= NOTE_MAP["content"],
            )

        content_text = content_node.get("text", None)
        title_text = None
        if content_node:
            # 获取上一个兄弟节点
            title_node = content_node.find_previous_sibling("node")
            if title_node:
                title_text = title_node.get("text", None)

        return title_text, content_text, author_text

    def _parse_note_comments():
        """解析笔记评论。"""
        soup = BeautifulSoup(d.dump_hierarchy(), "xml")
        # ,recursive=False 禁止递归查找子元素，也就是只查第一个
        comment_view =soup.find("node",attrs={"class":"androidx.recyclerview.widget.RecyclerView","drawing-order":"2"})

        if not comment_view:
            swiper_up()
            soup = BeautifulSoup(d.dump_hierarchy(), "xml")
            comment_view =soup.find("node",attrs={"class":"androidx.recyclerview.widget.RecyclerView","drawing-order":"2"},recursive=False)

        # print(comment_view)
        comments = comment_view.find_all("node", attrs={"class": "android.widget.LinearLayout"},recursive=False)
        results = []
        for comment in comments:
            items = comment.find_all("node",attrs={"class":"android.widget.TextView"})
            author = None
            content=None

            if items and len(items) >= 3:
                author = items[0].get("text", None)
                content = items[1].get("text", None)
                if content == "作者":
                    content = items[2].get("text", None)

                results.append({
                    "poster": author,
                    "content": content,
                })
        return results

    def parse_count(text: str):
        """
        解析:
        收藏 123
        收藏 1.2万
        """

        if not text:
            return 0

        match = re.search(r'([\d.]+)(万?)', text)

        if not match:
            return 0

        num = float(match.group(1))

        if match.group(2) == '万':
            num *= 10000

        return int(num)


    def _parse_likes_favorites_comments_count():
        """解析笔记点赞、收藏、评论数"""

        soup = BeautifulSoup(d.dump_hierarchy(), "xml")

        # 收藏
        favorites_elem = d.xpath(
            '//*[starts-with(@content-desc,"收藏")]'
        ).get()

        favorites_text = (
            favorites_elem.attrib.get("content-desc", "")
            if favorites_elem else ""
        )

        favorites_count = parse_count(favorites_text)

        # 点赞
        likes_elem = d.xpath(
            '//*[starts-with(@content-desc,"点赞")]'
        ).get()

        likes_text = (
            likes_elem.attrib.get("content-desc", "")
            if likes_elem else ""
        )

        likes_count = parse_count(likes_text)

        # 评论
        comments_elem = d.xpath(
            '//*[starts-with(@content-desc,"评论")]'
        ).get()

        comments_text = (
            comments_elem.attrib.get("content-desc", "")
            if comments_elem else ""
        )

        comments_count = parse_count(comments_text)

        return {
            "likes_count": likes_count,
            "favorites_count": favorites_count,
            "comments_count": comments_count
        }

    title_text, content_text, author_text = _parse_note_title_content_author()
    # comments = _parse_note_comments()

    if not title_text and not content_text:
        return None

    return {"title": title_text, "content": content_text, "author": author_text, **_parse_likes_favorites_comments_count()}


def click_note(nodes: List[Tag]):
    for node in nodes:
        if node.get("content-desc") and str(node.get("content-desc")).startswith("笔记"):
            view = node.find("node", attrs={"class": "android.view.View"})
            if not view:
                continue
            left, top, right, bottom = parse_bounds(str(view.get("bounds", None)))
            x = random.randint(left, right)
            y = random.randint(top, bottom)
            d.click(x, y)
            time.sleep(random.uniform(1, 2))
            print(parse_note())
            # next we should parse the comment or go to the next note

            time.sleep(random.uniform(2, 6))
            d.press("back")


while True:
    click_note(nodes)
    swiper_up()
    random.uniform(2, 6)
    swiper_up()
# parse_note()

AttributeError: 'NoneType' object has no attribute 'group'

In [ ]:
from red_note.main_page import RedNoteMainPage

red_note_main_page = RedNoteMainPage(d.dump_hierarchy())

In [ ]:
red_note_main_page.notes[1].model_dump()

In [ ]:
from red_note.note_detail import NodeDetailParser

detail = NodeDetailParser(d.dump_hierarchy())
detail.parse_note().model_dump()

In [ ]:
import time
import random
try:
    d(text='小红书', className='android.widget.TextView').click(timeout=10)
except u2.UiObjectNotFoundError:
    print("未找到小红书，继续执行...")
    exit(1)
time.sleep(random.randint(1, 3))
for el in d(textStartsWith="笔记", className="android.widget.FrameLayout").all():
    el.click()
    time.sleep(random.randint(1, 3))
    candidates = d(resourceId="com.xingin.xhs:id/0_resource_name_obfuscated", className="android.widget.TextView")
# 遍历实例，检查 drawingOrder
    for i in range(candidates.count):
        elem = candidates[i]
        if int(elem.info.get('drawing-order')) in [1,3]:
            print(elem.text)
            break
    else:
        print("未找到 drawing-order=3 的元素")
    d.press("back")
    time.sleep(random.randint(1, 3))


